# LP3 Machine Learning Practical 2 - Email Spam Detection using KNN and SVM

**Problem statement:** Classify the email using the binary classification method. Email spam detection has two states: a) Normal state - Not Spam, b) Abnormal state - Spam. Use K-Nearest Neighbors and Support Vector Machine for classification. Analyze their performance. Dataset: emails.csv from https://www.kaggle.com/datasets/balaka18/email-spam-classification-dataset-csv

In [ ]:
# Imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('/content/emails.csv')

In [ ]:
# Display first few rows of the dataset
print(df.head())

# Split dataset into features (X) and labels (y)
X = df.drop(columns=['Email No.', 'Prediction'])  # Features: word counts
y = df['Prediction']  # Target: spam (1) or not spam (0)

# Train-Test Split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize models
knn = KNeighborsClassifier(n_neighbors=5)
svm = SVC(kernel='linear', random_state=42, probability=True)  # probability=True for ROC/PR curves

# Train models
knn.fit(X_train, y_train)
svm.fit(X_train, y_train)

# Make predictions
y_pred_knn = knn.predict(X_test)
y_pred_svm = svm.predict(X_test)

# Evaluate models using accuracy, precision, recall, F1-score
def evaluate_model(y_true, y_pred, model_name):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    print(f"--- {model_name} ---")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")

    return accuracy, precision, recall, f1

# Evaluate both models
knn_metrics = evaluate_model(y_test, y_pred_knn, "K-Nearest Neighbors")
svm_metrics = evaluate_model(y_test, y_pred_svm, "Support Vector Machine")

In [ ]:
# Confusion Matrix for both models
def plot_confusion_matrix(y_true, y_pred, model_name):
    cm = confusion_matrix(y_true, y_pred)
    disp = sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Not Spam', 'Spam'], yticklabels=['Not Spam', 'Spam'])
    disp.set_title(f"{model_name} Confusion Matrix")
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

plot_confusion_matrix(y_test, y_pred_knn, "K-Nearest Neighbors")
plot_confusion_matrix(y_test, y_pred_svm, "Support Vector Machine")

In [ ]:
from sklearn.metrics import roc_curve, auc, precision_recall_curve

# ROC Curve and Precision-Recall Curve
def plot_roc_pr_curve(y_true, y_scores_knn, y_scores_svm):
    # ROC Curve
    fpr_knn, tpr_knn, _ = roc_curve(y_true, y_scores_knn)
    fpr_svm, tpr_svm, _ = roc_curve(y_true, y_scores_svm)

    auc_knn = auc(fpr_knn, tpr_knn)
    auc_svm = auc(fpr_svm, tpr_svm)

    # Plot ROC Curve
    plt.figure(figsize=(14, 6))
    plt.subplot(1, 2, 1)
    plt.plot(fpr_knn, tpr_knn, label=f'KNN (AUC = {auc_knn:.2f})')
    plt.plot(fpr_svm, tpr_svm, label=f'SVM (AUC = {auc_svm:.2f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.legend()

    # Precision-Recall Curve
    precision_knn, recall_knn, _ = precision_recall_curve(y_true, y_scores_knn)
    precision_svm, recall_svm, _ = precision_recall_curve(y_true, y_scores_svm)

    plt.subplot(1, 2, 2)
    plt.plot(recall_knn, precision_knn, label='KNN')
    plt.plot(recall_svm, precision_svm, label='SVM')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curve')
    plt.legend()

    plt.show()

# Get prediction probabilities for ROC/PR curves (SVM requires decision function or predict_proba)
y_scores_knn = knn.predict_proba(X_test)[:, 1]
y_scores_svm = svm.predict_proba(X_test)[:, 1]

plot_roc_pr_curve(y_test, y_scores_knn, y_scores_svm)

In [ ]:
# Comparing models: Bar chart of metrics
metrics_df = pd.DataFrame([knn_metrics, svm_metrics], columns=['Accuracy', 'Precision', 'Recall', 'F1-Score'], index=['KNN', 'SVM'])
metrics_df.plot(kind='bar', figsize=(10, 6))
plt.title("Comparison of KNN and SVM Performance Metrics")
plt.ylabel('Scores')
plt.xticks(rotation=0)
plt.show()

## Theory and Explanation
The last cell contains the theory, steps, result interpretation and viva questions as comments.

In [ ]:
# =====================================================================
# THEORY AND EXPLANATION - Practical 2: Email Spam Detection (KNN and SVM)
# =====================================================================
#
# 1. PROBLEM
# Classify each email as Not Spam (0, the normal state) or Spam (1, the abnormal
# state). Only two classes, so this is BINARY CLASSIFICATION (supervised learning).
#
# 2. DATASET
# Each row is one email. Columns are: Email No. (an id, dropped), 3000 columns
# holding the count of a particular word in that email, and Prediction (the label).
# So the features are word counts (a bag-of-words representation) and the dataset
# is already numeric. That is why TfidfVectorizer was imported but never needed.
#
# 3. KEY THEORY
# a) K-Nearest Neighbors (KNN)
#    A lazy learner: it does no training, it just stores the data. To classify a new
#    email it computes the distance (Euclidean) to every training email, takes the
#    k = 5 nearest ones and predicts the majority class among them.
#    Weaknesses: slow at prediction time, sensitive to feature scale, and weak in
#    very high dimensions (3000 features here) because distances become less
#    meaningful (curse of dimensionality).
#
# b) Support Vector Machine (SVM)
#    Finds the hyperplane that separates the two classes with the MAXIMUM margin.
#    The training points that lie on the margin are the support vectors, and only
#    they define the boundary. A linear kernel is used here, which works very well
#    for high-dimensional sparse text data. With other kernels (RBF, polynomial) the
#    kernel trick lets SVM separate non-linear data. The parameter C controls the
#    trade-off between a wide margin and misclassified points.
#    probability=True makes sklearn fit an extra calibration step (Platt scaling) so
#    that predict_proba can be used for the ROC and PR curves. It slows training.
#
# c) Confusion matrix terms (spam is the positive class)
#    TP: spam predicted as spam           TN: normal predicted as normal
#    FP: normal predicted as spam         FN: spam predicted as normal
#    Accuracy  = (TP + TN) / total
#    Precision = TP / (TP + FP)   Of the emails flagged spam, how many really are.
#    Recall    = TP / (TP + FN)   Of all real spam, how many were caught.
#    F1-score  = 2 * P * R / (P + R)   Harmonic mean of precision and recall.
#
# d) ROC curve and AUC
#    ROC plots True Positive Rate against False Positive Rate for every probability
#    threshold. AUC is the area under it (1 is perfect, 0.5 is random guessing).
#    The Precision-Recall curve plots precision against recall and is more
#    informative when classes are imbalanced (spam is the minority here).
#
# 4. STEPS (code flow)
#    1. Load emails.csv, drop Email No. and split into X (word counts) and y.
#    2. 80/20 train-test split (random_state = 42). Test set has 1035 emails.
#    3. Create KNN (k = 5) and SVM (linear kernel), fit both on the training data.
#    4. Predict on the test data and compute accuracy, precision, recall, F1.
#    5. Plot confusion matrices, then ROC and Precision-Recall curves.
#    6. Plot a bar chart comparing the metrics.
#
# 5. RESULTS AND HOW TO READ THEM
#                       KNN (k=5)     SVM (linear)
#    Accuracy            0.8628         0.9594
#    Precision           0.7265         0.9205
#    Recall              0.8345         0.9392
#    F1-score            0.7767         0.9298
#    AUC (from plot)     0.94           0.98
#
#    Confusion matrices (rows = actual, columns = predicted):
#    KNN : TN 646, FP 93,  FN 49, TP 247
#    SVM : TN 715, FP 24,  FN 18, TP 278
#
#    SVM is better on every metric. KNN wrongly marked 93 normal emails as spam
#    (false positives), which is the costly mistake for a real mail user because
#    important mail would go to the spam folder. SVM made only 24 such mistakes
#    and missed only 18 spam emails against 49 for KNN.
#    Why SVM wins: with 3000 sparse word-count features, a linear boundary
#    separates spam well, while KNN suffers from the curse of dimensionality and
#    from unscaled counts dominating the distance.
#
# 6. VIVA QUESTIONS
#    Q: What is the normal and abnormal state here?
#    A: Normal = Not Spam (0), abnormal = Spam (1).
#    Q: Why is accuracy alone not enough?
#    A: Classes are imbalanced (fewer spam emails). Precision, recall and F1 show
#       what kind of mistakes are made.
#    Q: Which is worse here, a false positive or a false negative?
#    A: Usually a false positive (a real email lost in spam), so precision matters.
#    Q: What is a support vector?
#    A: A training point lying on the margin that defines the decision boundary.
#    Q: How do you choose k in KNN?
#    A: Try several values and pick the one with the lowest validation error,
#       ideally using cross-validation. An odd k avoids ties.
#    Q: Why does KNN need scaling?
#    A: Distance is dominated by features with larger numeric ranges.